<a href="https://colab.research.google.com/github/JorgeSanchezCano/curso-llama/blob/main/Copia_de_Hands_On_Fine_Tuning_con_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [1]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.8 MB/s eta 0:00:00
Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [2]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [3]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no puedes cambiar tu pedido.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [4]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "
     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},
    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},
    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},
    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [6]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base

!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()
# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [7]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.554', 'grad_norm': '1.737', 'learning_rate': '0.0002', 'entropy': '2.25', 'num_tokens': '229', 'mean_token_accuracy': '0.5', 'epoch': '1'}
{'loss': '2.512', 'grad_norm': '1.753', 'learning_rate': '0.0001933', 'entropy': '2.244', 'num_tokens': '458', 'mean_token_accuracy': '0.4955', 'epoch': '2'}
{'loss': '2.456', 'grad_norm': '1.926', 'learning_rate': '0.0001867', 'entropy': '2.234', 'num_tokens': '687', 'mean_token_accuracy': '0.5089', 'epoch': '3'}
{'loss': '2.387', 'grad_norm': '2.05', 'learning_rate': '0.00018', 'entropy': '2.223', 'num_tokens': '916', 'mean_token_accuracy': '0.5268', 'epoch': '4'}
{'loss': '2.317', 'grad_norm': '1.984', 'learning_rate': '0.0001733', 'entropy': '2.208', 'num_tokens': '1145', 'mean_token_accuracy': '0.5223', 'epoch': '5'}
{'loss': '2.246', 'grad_norm': '2.011', 'learning_rate': '0.0001667', 'entropy': '2.188', 'num_tokens': '1374', 'mean_token_accuracy': '0.5446', 'epoch': '6'}
{'loss': '2.17', 'grad_norm': '2.193', 'learning_rate': '0.0

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.

In [10]:
# Comparar la pérdidas
perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")


Pérdida al inicio del entrenamiento: 2.55
Pérdida final del entrenamiento: 1.77
Reducción: 31%


In [11]:
# Referencia cualitativa vs Referencia con un modelo de producción
!pip install groq -q
from groq import Groq

respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("Respuesta del modelo ajustado (TinyLlama + LoRA):\n", respuesta_ajustada)

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_prueba + " Respuesta muy breve y corta."}]
)

print("\nRespuesta de referencia (Groq / GPT-OSS-20B):\n", response.choices[0].message.content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.3 MB/s eta 0:00:00
Respuesta del modelo ajustado (TinyLlama + LoRA):
 Sleep, and the client.

Respuesta de referencia (Groq / GPT-OSS-20B):
 No, no se puede.


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.

## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva. En esta solución se usa como ejemplo un asistente de dudas frecuentes del propio curso.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías, inicia sesión en Hugging Face con tu token y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [12]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets
!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


In [13]:
# Cargar el modelo base de Llama y su tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [14]:
# Definir la funcion de generacion de texto
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()



In [15]:
# Definir la lista ejemplos y convertirla en dataset
from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Tiene costo el envio si compro 500 pesos?\nAgente: No, El envío "
     "Es gratis despues de los 250 pesos de pedido"},
    {"texto": "Cliente: ¿Cual es el precio de los Tacos?\nAgente: El precio es de 12 pesos por taco."},
    {"texto": "Cliente: ¿Manejan Alguna promoción?\nAgente: Sí, Los dias Martes y Miercoles son al 2 por el precio de 1"},
    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo "},
    {"texto": "Cliente: ¿Puedo Pagar con transferencia?\nAgente: Si antes de hacer el envio para que el repartidor tenga es información"},
]

dataset = Dataset.from_list(ejemplos)
dataset

Dataset({
    features: ['texto'],
    num_rows: 5
})

**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [16]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base
prompt_prueba = "Cliente: ¿Tiene costo el envio si compro 500 pesos?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no tiene costo del envio.


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [17]:
# Configurar LoraConfig y aplicarlo al modelo base
!pip uninstall -y torchao --quiet

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [19]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=30,
    per_device_train_batch_size=5,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)
perdida_final = resultado_entrenamiento.training_loss

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '1.674', 'grad_norm': '2.174', 'learning_rate': '0.0002', 'entropy': '2.149', 'num_tokens': '199', 'mean_token_accuracy': '0.6701', 'epoch': '1'}
{'loss': '1.552', 'grad_norm': '2.14', 'learning_rate': '0.0001933', 'entropy': '2.059', 'num_tokens': '398', 'mean_token_accuracy': '0.6701', 'epoch': '2'}
{'loss': '1.437', 'grad_norm': '2.079', 'learning_rate': '0.0001867', 'entropy': '1.967', 'num_tokens': '597', 'mean_token_accuracy': '0.6856', 'epoch': '3'}
{'loss': '1.328', 'grad_norm': '2.107', 'learning_rate': '0.00018', 'entropy': '1.88', 'num_tokens': '796', 'mean_token_accuracy': '0.701', 'epoch': '4'}
{'loss': '1.231', 'grad_norm': '2.223', 'learning_rate': '0.0001733', 'entropy': '1.799', 'num_tokens': '995', 'mean_token_accuracy': '0.7216', 'epoch': '5'}
{'loss': '1.136', 'grad_norm': '2.365', 'learning_rate': '0.0001667', 'entropy': '1.718', 'num_tokens': '1194', 'mean_token_accuracy': '0.7474', 'epoch': '6'}
{'loss': '1.048', 'grad_norm': '2.567', 'learning_rate': '0

**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [20]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva
perdida_inicial = trainer.state.log_history[0]['loss']
print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

Pérdida al inicio del entrenamiento: 1.67
Pérdida final del entrenamiento: 0.64
Reducción: 62%


In [ ]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico):
